In [ ]:
!pip install -q streamlit pyjwt bcrypt pyngrok

In [ ]:
import os
from google.colab import userdata

# Create the Streamlit hidden directory
os.makedirs(".streamlit", exist_ok=True)

# Write the Colab secrets into Streamlit's secrets file
with open(".streamlit/secrets.toml", "w") as f:
    f.write(f'JWT_SECRET = "{userdata.get("JWT_SECRET")}"\n')
    f.write(f'EMAIL_ADDRESS = "{userdata.get("EMAIL_ADDRESS")}"\n')
    f.write(f'EMAIL_PASSWORD = "{userdata.get("EMAIL_PASSWORD")}"\n')

print("Secrets successfully bridged to Streamlit!")

In [ ]:
%%writefile app.py
import streamlit as st
import sqlite3
import re
import random
import smtplib
import jwt
from datetime import datetime, timedelta, timezone
from email.mime.text import MIMEText
import bcrypt

# ==========================================
# 1. PAGE SETUP & INTENTIONAL MARITIME STYLING
# ==========================================
st.set_page_config(
    page_title="FreightQuote AI - Auth",
    page_icon="⚓",
    layout="centered",
    initial_sidebar_state="collapsed"
)

# Maritime-inspired professional styling (no decorative gradients/glassmorphism)
st.markdown("""
<style>
    .stApp {
        background-color: #0B192C;
        color: #F8FAFC;
    }
    div[data-testid="stForm"] {
        background-color: #1E293B;
        border: 1px solid #334155;
        border-radius: 8px;
        padding: 24px;
    }
    h1, h2, h3 {
        color: #F8FAFC !important;
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
        font-weight: 600;
    }
    .stTextInput > label, .stSelectbox > label {
        color: #CBD5E1 !important;
        font-size: 14px;
        font-weight: 500;
    }
    .stButton > button {
        background-color: #0284C7;
        color: #FFFFFF;
        border: none;
        border-radius: 6px;
        font-weight: 600;
        width: 100%;
        padding: 0.5rem 1rem;
        transition: background-color 0.15s ease-in-out;
    }
    .stButton > button:hover {
        background-color: #0369A1;
        color: #FFFFFF;
    }
    .stAlert {
        border-radius: 6px;
    }
    hr {
        border-color: #334155;
    }
</style>
""", unsafe_allow_html=True)

# ==========================================
# 2. DATABASE LAYER (SQLite)
# ==========================================
DB_FILE = "freightquote_users.db"

def init_db():
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    c.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            email TEXT UNIQUE NOT NULL,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

init_db()

# ==========================================
# 3. SECURITY & CRYPTO HELPERS
# ==========================================
# Fixed Admin credentials defined directly in code as required by Step 11
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD_HASH = bcrypt.hashpw(b"Admin@FreightQuote2026", bcrypt.gensalt()).decode()

def hash_value(value: str) -> str:
    return bcrypt.hashpw(value.encode(), bcrypt.gensalt()).decode()

def verify_hash(value: str, hashed: str) -> bool:
    try:
        return bcrypt.checkpw(value.encode(), hashed.encode())
    except Exception:
        return False

def generate_jwt(username: str, email: str, role: str, secret: str) -> str:
    payload = {
        "sub": username,
        "email": email,
        "role": role,
        "exp": datetime.now(timezone.utc) + timedelta(hours=2)
    }
    return jwt.encode(payload, secret, algorithm="HS256")

def decode_jwt(token: str, secret: str):
    try:
        return jwt.decode(token, secret, algorithms=["HS256"])
    except (jwt.ExpiredSignatureError, jwt.InvalidTokenError):
        return None

# ==========================================
# 4. FIELD VALIDATION (Step 8 Rules)
# ==========================================
def validate_email(email: str) -> bool:
    # Rule: >=2 chars before @, >=2 chars between @ and dot, >=2 chars after dot
    pattern = r"^[a-zA-Z0-9_.+-]{2,}@[a-zA-Z0-9-]{2,}\.[a-zA-Z0-9-.]{2,}$"
    return bool(re.match(pattern, email.strip()))

def validate_password(password: str) -> tuple[bool, str]:
    if len(password) < 8:
        return False, "Password must be at least 8 characters long."
    if not re.search(r"[A-Z]", password):
        return False, "Password must contain at least one uppercase letter."
    if not re.search(r"[a-z]", password):
        return False, "Password must contain at least one lowercase letter."
    if not re.search(r"[0-9]", password):
        return False, "Password must contain at least one number."
    if not re.search(r"[!@#$%^&*(),.?\":{}|<>]", password):
        return False, "Password must contain at least one special symbol."
    return True, ""

# ==========================================
# 5. EMAIL OTP SERVICE (Gmail SMTP)
# ==========================================
def send_otp_email(to_email: str, otp_code: str, sender_email: str, app_password: str) -> bool:
    msg = MIMEText(f"Your FreightQuote AI one-time verification code is: {otp_code}\n\nThis code expires in 10 minutes.")
    msg["Subject"] = "FreightQuote AI - Verification OTP"
    msg["From"] = sender_email
    msg["To"] = to_email

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(sender_email, app_password)
            server.sendmail(sender_email, [to_email], msg.as_string())
        return True
    except Exception as e:
        st.error(f"Failed to transmit OTP email: {str(e)}")
        return False

# ==========================================
# 6. SESSION STATE INITIALIZATION
# ==========================================
if "current_view" not in st.session_state:
    st.session_state.current_view = "login"
if "jwt_token" not in st.session_state:
    st.session_state.jwt_token = None
if "otp_store" not in st.session_state:
    st.session_state.otp_store = {}

# Retrieve secrets directly from Streamlit's secrets manager
try:
    JWT_SECRET = st.secrets["JWT_SECRET"]
    EMAIL_ADDRESS = st.secrets["EMAIL_ADDRESS"]
    EMAIL_PASSWORD = st.secrets["EMAIL_PASSWORD"]
except Exception as e:
    st.error("Failed to load Streamlit secrets. Please run the secrets bridge cell.")
    st.stop()

# Fixed list of security questions
SECURITY_QUESTIONS = [
    "What was the name of your first school?",
    "What is your mother's maiden name?",
    "In what city were you born?",
    "What was your childhood nickname?"
]

# ==========================================
# 7. ROUTING AND VIEW LOGIC
# ==========================================
session_data = decode_jwt(st.session_state.jwt_token, JWT_SECRET) if st.session_state.jwt_token else None

# Check authentication state
if session_data:
    if session_data.get("role") == "admin":
        st.session_state.current_view = "admin_dashboard"
    else:
        st.session_state.current_view = "user_dashboard"

# Navigation bar for unauthenticated views
if not session_data:
    cols = st.columns(4)
    if cols[0].button("Sign In"):
        st.session_state.current_view = "login"
        st.rerun()
    if cols[1].button("Create Account"):
        st.session_state.current_view = "signup"
        st.rerun()
    if cols[2].button("Forgot Password"):
        st.session_state.current_view = "forgot_password"
        st.rerun()
    if cols[3].button("Admin Portal"):
        st.session_state.current_view = "admin_login"
        st.rerun()
    st.markdown("---")

# VIEW: SIGNUP
if st.session_state.current_view == "signup":
    st.subheader("FreightQuote AI - Create Account")
    with st.form("signup_form"):
        username = st.text_input("Username*")
        email = st.text_input("Email Address*")
        password = st.text_input("Password*", type="password")
        confirm_password = st.text_input("Confirm Password*", type="password")
        sec_question = st.selectbox("Security Question*", SECURITY_QUESTIONS)
        sec_answer = st.text_input("Security Answer*", type="password")
        submitted = st.form_submit_button("Register")

        if submitted:
            if not all([username, email, password, confirm_password, sec_answer]):
                st.error("All fields are mandatory.")
            elif not validate_email(email):
                st.error("Invalid email address format.")
            elif password != confirm_password:
                st.error("Password and Confirm Password do not match.")
            else:
                is_valid, msg = validate_password(password)
                if not is_valid:
                    st.error(msg)
                else:
                    conn = sqlite3.connect(DB_FILE)
                    c = conn.cursor()
                    c.execute("SELECT id FROM users WHERE username = ? OR email = ?", (username.strip(), email.strip()))
                    existing = c.fetchone()
                    if existing:
                        st.error("Username or email already exists. Please choose another.")
                        conn.close()
                    else:
                        pw_hash = hash_value(password)
                        ans_hash = hash_value(sec_answer.strip().lower())
                        c.execute("""
                            INSERT INTO users (username, email, password_hash, security_question, security_answer_hash)
                            VALUES (?, ?, ?, ?, ?)
                        """, (username.strip(), email.strip(), pw_hash, sec_question, ans_hash))
                        conn.commit()
                        conn.close()
                        st.success("Account created successfully. Redirecting to login...")
                        st.session_state.current_view = "login"
                        st.rerun()

# VIEW: LOGIN
elif st.session_state.current_view == "login":
    st.subheader("FreightQuote AI - Maritime Portal Login")
    with st.form("login_form"):
        identifier = st.text_input("Username or Email*")
        password = st.text_input("Password*", type="password")
        submitted = st.form_submit_button("Sign In")

        if submitted:
            if not identifier or not password:
                st.error("All fields are mandatory.")
            else:
                conn = sqlite3.connect(DB_FILE)
                c = conn.cursor()
                c.execute("SELECT username, email, password_hash FROM users WHERE username = ? OR email = ?", (identifier.strip(), identifier.strip()))
                user = c.fetchone()
                conn.close()

                # Step 3 Rule: Generic error message, never specify username vs password failure
                if user and verify_hash(password, user[2]):
                    st.session_state.jwt_token = generate_jwt(user[0], user[1], "user", JWT_SECRET)
                    st.rerun()
                else:
                    st.error("Invalid username/email or password.")

# VIEW: ADMIN LOGIN
elif st.session_state.current_view == "admin_login":
    st.subheader("System Administrator Access")
    with st.form("admin_login_form"):
        admin_user = st.text_input("Admin Username*")
        admin_pass = st.text_input("Admin Password*", type="password")
        submitted = st.form_submit_button("Authenticate Admin")

        if submitted:
            if admin_user == ADMIN_USERNAME and verify_hash(admin_pass, ADMIN_PASSWORD_HASH):
                st.session_state.jwt_token = generate_jwt(ADMIN_USERNAME, "admin@freightquote.local", "admin", JWT_SECRET)
                st.rerun()
            else:
                st.error("Invalid administrator credentials.")

# VIEW: FORGOT PASSWORD (DUAL-ROUTE)
elif st.session_state.current_view == "forgot_password":
    st.subheader("Account Recovery")
    recovery_method = st.radio("Select Recovery Route", ["Security Question", "Email OTP"], horizontal=True)

    if recovery_method == "Security Question":
        with st.form("sec_question_form"):
            user_target = st.text_input("Registered Username*")
            sec_answer_input = st.text_input("Your Security Answer*", type="password")
            new_pw = st.text_input("New Password*", type="password")
            confirm_new_pw = st.text_input("Confirm New Password*", type="password")
            reset_btn = st.form_submit_button("Reset Password via Security Question")

            if reset_btn:
                if not all([user_target, sec_answer_input, new_pw, confirm_new_pw]):
                    st.error("All fields are mandatory.")
                elif new_pw != confirm_new_pw:
                    st.error("Passwords do not match.")
                else:
                    is_valid, msg = validate_password(new_pw)
                    if not is_valid:
                        st.error(msg)
                    else:
                        conn = sqlite3.connect(DB_FILE)
                        c = conn.cursor()
                        c.execute("SELECT id, security_answer_hash FROM users WHERE username = ?", (user_target.strip(),))
                        record = c.fetchone()
                        if record and verify_hash(sec_answer_input.strip().lower(), record[1]):
                            new_hash = hash_value(new_pw)
                            c.execute("UPDATE users SET password_hash = ? WHERE id = ?", (new_hash, record[0]))
                            conn.commit()
                            conn.close()
                            st.success("Password reset successful. Please sign in.")
                            st.session_state.current_view = "login"
                            st.rerun()
                        else:
                            conn.close()
                            st.error("Verification failed. Please check your credentials.")

    elif recovery_method == "Email OTP":
        col1, col2 = st.columns([3, 1])
        target_email = col1.text_input("Registered Email Address")
        send_otp_btn = col2.button("Send Code", use_container_width=True)

        if send_otp_btn:
            if not target_email or not validate_email(target_email):
                st.error("Enter a valid email address.")
            else:
                conn = sqlite3.connect(DB_FILE)
                c = conn.cursor()
                c.execute("SELECT id FROM users WHERE email = ?", (target_email.strip(),))
                exists = c.fetchone()
                conn.close()

                if exists:
                    otp_code = str(random.randint(100000, 999999))
                    st.session_state.otp_store[target_email.strip()] = {
                        "code": otp_code,
                        "expiry": datetime.now() + timedelta(minutes=10)
                    }
                    if send_otp_email(target_email.strip(), otp_code, EMAIL_ADDRESS, EMAIL_PASSWORD):
                        st.success("Verification code dispatched to your email.")
                else:
                    st.error("Email not found in registered accounts.")

        with st.form("otp_reset_form"):
            otp_val = st.text_input("Enter 6-Digit OTP*")
            otp_new_pw = st.text_input("New Password*", type="password")
            otp_confirm_pw = st.text_input("Confirm New Password*", type="password")
            confirm_reset_btn = st.form_submit_button("Verify & Reset Password")

            if confirm_reset_btn:
                if not all([target_email, otp_val, otp_new_pw, otp_confirm_pw]):
                    st.error("All fields are mandatory.")
                elif otp_new_pw != otp_confirm_pw:
                    st.error("Passwords do not match.")
                else:
                    is_valid, msg = validate_password(otp_new_pw)
                    if not is_valid:
                        st.error(msg)
                    else:
                        cached = st.session_state.otp_store.get(target_email.strip())
                        if cached and cached["code"] == otp_val.strip() and datetime.now() <= cached["expiry"]:
                            conn = sqlite3.connect(DB_FILE)
                            c = conn.cursor()
                            c.execute("UPDATE users SET password_hash = ? WHERE email = ?", (hash_value(otp_new_pw), target_email.strip()))
                            conn.commit()
                            conn.close()
                            del st.session_state.otp_store[target_email.strip()]
                            st.success("Password reset verified successfully. Please log in.")
                            st.session_state.current_view = "login"
                            st.rerun()
                        else:
                            st.error("Invalid or expired verification code.")

# VIEW: USER DASHBOARD (Step 10)
elif st.session_state.current_view == "user_dashboard":
    st.title("⚓ FreightQuote AI — Broker Portal")
    st.write(f"Logged in as: **{session_data['sub']}** (`{session_data['email']}`)")
    st.caption("Active Session Token Verified · Role: Logistics Agent")
    st.markdown("---")

    st.info("System operational. The 7-Agent Freight Intelligence Engine is currently initializing for Milestone 2.")

    if st.button("Sign Out"):
        st.session_state.jwt_token = None
        st.session_state.current_view = "login"
        st.rerun()

# VIEW: ADMIN DASHBOARD (Step 11)
elif st.session_state.current_view == "admin_dashboard":
    st.title("🛡️ FreightQuote AI — Admin Directory")
    st.caption("Protected Administration Layer · Confidential User Directory")
    st.markdown("---")

    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    # Step 11: Display usernames and emails only — passwords are never exposed
    c.execute("SELECT id, username, email, created_at FROM users ORDER BY id DESC")
    records = c.fetchall()
    conn.close()

    if records:
        table_data = [{"ID": r[0], "Username": r[1], "Email": r[2], "Registration Timestamp": r[3]} for r in records]
        st.dataframe(table_data, use_container_width=True)
    else:
        st.info("No registered users found.")

    if st.button("Exit Admin Session"):
        st.session_state.jwt_token = None
        st.session_state.current_view = "login"
        st.rerun()

In [ ]:
import os
import time
from pyngrok import ngrok, conf
from google.colab import userdata

# 1. Terminate any previous background tunnels/Streamlit instances
ngrok.kill()
!pkill -f "streamlit"

# 2. Authenticate ngrok using Colab Secret
ngrok_token = userdata.get("NGROK_AUTHTOKEN")
conf.get_default().auth_token = ngrok_token

# 3. Launch Streamlit in background
!nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
time.sleep(3)

# 4. Open public tunnel
public_tunnel = ngrok.connect(8501, "http")
print("\n" + "="*60)
print("⚓ FreightQuote AI Authentication Portal is LIVE:")
print(f"👉 URL: {public_tunnel.public_url}")
print("="*60 + "\n")